In [0]:
use sql_catalog.sql_schema;
select * from emp;

empno,ename,job,mgr,hiredate,sal,comm,deptno
7369,SMITH,CLERK,7902,1980-12-17,800,null,20
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30
7521,WARD,SALESMAN,7698,1981-02-22,1250,500,30
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7654,MARTIN,SALESMAN,7698,1981-09-28,1250,1400,30
7698,BLAKE,MANAGER,7839,1981-05-01,2850,null,30
7782,CLARK,MANAGER,7839,1981-06-09,2450,null,10
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20
7839,KING,PRESIDENT,null,1981-11-17,5000,null,10
7844,TURNER,SALESMAN,7698,1981-09-08,1500,0,30


In [0]:
-- [Easy] List the employees and their manager names.
select e.ename,m.ename as manager
from emp e 
join emp m
on e.mgr=m.empno;

--  When you execute JOIN ... ON e.mgr = m.empno, SQL creates a temporary combined row behind the scenes:

--         e                       m
-- empno ename  mgr         empno   ename mgr 
-- 7369  SMITH  7902        7902    FORD  7566
-- 7499  ALLEN  7698        7566    JONES 7839


ename,manager
SMITH,FORD
ALLEN,BLAKE
WARD,BLAKE
JONES,KING
MARTIN,BLAKE
BLAKE,KING
CLARK,KING
SCOTT,JONES
TURNER,BLAKE
ADAMS,SCOTT


In [0]:
-- If you want to include top-level employees who do not have a manager, switch to a LEFT JOIN:
SELECT e.ename AS employee, m.ename AS manager
       --COALESCE(m.ename, 'No Manager') AS manager
FROM emp e
LEFT JOIN emp m
  ON e.mgr = m.empno;

employee,manager
SMITH,FORD
ALLEN,BLAKE
WARD,BLAKE
JONES,KING
MARTIN,BLAKE
BLAKE,KING
CLARK,KING
SCOTT,JONES
KING,null
TURNER,BLAKE


In [0]:
-- [Easy] List the employees whose manager name is JONES and also list their manager name.

select e.ename,m.ename as manager
from emp e 
join emp m
on e.mgr=m.empno
where m.ename='JONES';



ename,manager
SCOTT,JONES
FORD,JONES


In [0]:
-- [Easy] List the employees whose salaries are greater than the salaries of their managers.

select e.ename,e.sal, m.ename as manager, m.sal as mgr_sal
from emp e
join emp m
on e.mgr=m.empno
where e.sal>m.sal;

ename,sal,manager,mgr_sal
SCOTT,3000,JONES,2975
FORD,3000,JONES,2975


In [0]:
-- [Easy] List those managers who are getting less than their employees' salary.
select m.ename as manager, e.ename, m.sal as mgr_sal, e.sal as emp_sal
from emp e
join emp m
on e.mgr=m.empno
where m.sal<e.sal;


manager,ename,mgr_sal,emp_sal
JONES,SCOTT,2975,3000
JONES,FORD,2975,3000


In [0]:
-- [Easy] List the managers who are not working under the president.

-- WRONG approach
select ename, job 
from emp
where job='MANAGER' and ename not in (select ename from emp where job='PRESIDENT')

ename,job
JONES,MANAGER
BLAKE,MANAGER
CLARK,MANAGER


In [0]:
-- Approach 1: Using Subquery on mgr (Recommended)
-- Find all managers whose mgr ID does not belong to the President:
SELECT empno 
      FROM emp 
      WHERE UPPER(job) = 'PRESIDENT'

empno
7839


In [0]:
SELECT ename, job 
FROM emp
WHERE job = 'MANAGER' 
  AND mgr NOT IN (
      SELECT empno 
      FROM emp 
      WHERE UPPER(job) = 'PRESIDENT'
  );

ename,job


In [0]:
--Approach 2: Using Self Join
--Join the employee table to itself to compare the employee's manager's job:
SELECT e.ename, e.job 
FROM emp e
JOIN emp m ON e.mgr = m.empno
WHERE e.job = 'MANAGER' 
  AND m.job != 'PRESIDENT';

ename,job


In [0]:
--  [Easy] Print the details of all employees who are subordinates to BLAKE.

select e.ename, m.ename as manager
from emp e 
join emp m
on e.mgr=m.empno
where m.ename='BLAKE'

ename,manager
ALLEN,BLAKE
WARD,BLAKE
MARTIN,BLAKE
TURNER,BLAKE
JAMES,BLAKE


In [0]:
select avg(sal),mgr
from emp
group by mgr;
--Your query also returns one extra row for mgr = NULL. 
--That is KING's own row, since he has no manager. Filter it out with WHERE mgr IS NOT NULL.

avg(sal),mgr
800.0,7902
1310.0,7698
2758.3333333333335,7839
3000.0,7566
5000.0,null
1100.0,7788
1300.0,7782


In [0]:
    SELECT mgr, AVG(sal) AS avg_sal
    FROM emp
    WHERE mgr IS NOT NULL
    GROUP BY mgr


mgr,avg_sal
7902,800.0
7698,1310.0
7839,2758.3333333333335
7566,3000.0
7788,1100.0
7782,1300.0


In [0]:
SELECT m.ename, m.sal, t.avg_sal
FROM emp m
JOIN (
    SELECT mgr, AVG(sal) AS avg_sal
    FROM emp
    WHERE mgr IS NOT NULL
    GROUP BY mgr
) t
  ON m.empno = t.mgr
WHERE m.sal > t.avg_sal;

ename,sal,avg_sal
BLAKE,2850,1310.0
CLARK,2450,1300.0
SCOTT,3000,1100.0
KING,5000,2758.3333333333335
FORD,3000,800.0


In [0]:
--  [Hard] List the managers whose salary is more than their employees' average salary.

select avg(sal)
from emp
group by mgr;

SELECT m.empno AS manager_id,
       m.ename AS manager_name,
       m.sal AS manager_sal,
       AVG(e.sal) AS direct_reports_avg_sal
FROM emp m
JOIN emp e ON m.empno = e.mgr
GROUP BY m.empno, m.ename, m.sal
HAVING m.sal > AVG(e.sal);


manager_id,manager_name,manager_sal,direct_reports_avg_sal
7698,BLAKE,2850,1310.0
7839,KING,5000,2758.33
7902,FORD,3000,800.0
7782,CLARK,2450,1300.0
7788,SCOTT,3000,1100.0


In [0]:
-- Alternative: self join with HAVING
SELECT m.ename, m.sal, AVG(e.sal) AS avg_sal
FROM emp m
JOIN emp e ON e.mgr = m.empno
GROUP BY m.empno, m.ename, m.sal
HAVING m.sal > AVG(e.sal);

ename,sal,avg_sal
BLAKE,2850,1310.0
KING,5000,2758.3333333333335
FORD,3000,800.0
CLARK,2450,1300.0
SCOTT,3000,1100.0


In [0]:
select avg(sal),deptno
from emp
group by deptno

avg(sal),deptno
2175.0,20
1566.6666666666667,30
2916.6666666666665,10


In [0]:
select avg(e.sal)
from emp e
join emp m
on e.mgr=m.empno

avg(e.sal)
1848.076923076923


In [0]:
-- [Hard] List the employees whose salary is less than their manager's salary but more than any other manager's salary.
select e.ename , m.ename as manager, e.sal as employee_sal, m.sal as manager_sal
from emp e
join emp m
on e.mgr=m.empno
where e.sal < m.sal and e.sal > (select


ename,manager,employee_sal,manager_sal
SMITH,FORD,800,3000
ALLEN,BLAKE,1600,2850
WARD,BLAKE,1250,2850
JONES,KING,2975,5000
MARTIN,BLAKE,1250,2850
BLAKE,KING,2850,5000
CLARK,KING,2450,5000
TURNER,BLAKE,1500,2850
ADAMS,SCOTT,1100,3000
JAMES,BLAKE,950,2850


In [0]:
SELECT e.ename,
       m.ename AS manager,
       e.sal   AS employee_sal,
       m.sal   AS manager_sal
FROM emp e
JOIN emp m
  ON e.mgr = m.empno
WHERE e.sal < m.sal
  AND e.sal > (
        SELECT MIN(sal)
        FROM emp
        WHERE empno IN (SELECT mgr FROM emp WHERE mgr IS NOT NULL)  -- people who manage someone
          AND empno <> m.empno                                      -- exclude the employee's own manager
      );

ename,manager,employee_sal,manager_sal
BLAKE,KING,2850,5000
JONES,KING,2975,5000


In [0]:
SELECT mgr FROM emp WHERE mgr IS NOT NULL

mgr
7902
7698
7698
7839
7698
7839
7839
7566
7698
7788


In [0]:
sELECT e.ename,
       m.ename AS manager,
       e.sal   AS employee_sal,
       m.sal   AS manager_sal
FROM emp e
JOIN emp m
  ON e.mgr = m.empno
WHERE e.sal < m.sal
  AND e.sal > (
        SELECT MIN(sal)
        FROM emp
        WHERE empno IN (select mgr from emp)  -- people who manage someone
          AND empno <> m.empno                                      -- exclude the employee's own manager
      );

ename,manager,employee_sal,manager_sal
BLAKE,KING,2850,5000
JONES,KING,2975,5000


In [0]:
--  [Hard] Find out the employees who joined the company before their Manager

select e.ename, e.hiredate, m.ename as manager,m.hiredate as man_hiredate
from emp e 
join emp m 
on e.mgr = m.empno

where e.hiredate < m.hiredate



ename,hiredate,manager,man_hiredate
SMITH,1980-12-17,FORD,1981-12-03
ALLEN,1981-02-20,BLAKE,1981-05-01
WARD,1981-02-22,BLAKE,1981-05-01
JONES,1981-04-02,KING,1981-11-17
BLAKE,1981-05-01,KING,1981-11-17
CLARK,1981-06-09,KING,1981-11-17


In [0]:
SELECT e.ename AS emp,
       e.hiredate AS emp_hiredate,
       m.ename AS manager,
       m.hiredate AS mgr_hiredate
FROM emp e
JOIN emp m
  ON e.mgr = m.empno
WHERE e.hiredate < m.hiredate;

emp,emp_hiredate,manager,mgr_hiredate
SMITH,1980-12-17,FORD,1981-12-03
ALLEN,1981-02-20,BLAKE,1981-05-01
WARD,1981-02-22,BLAKE,1981-05-01
JONES,1981-04-02,KING,1981-11-17
BLAKE,1981-05-01,KING,1981-11-17
CLARK,1981-06-09,KING,1981-11-17
